In [2]:

def more_general(h1, h2):
    """True if h1 is more general than or equal to h2 (h1 covers everything h2 covers)."""
    for x, y in zip(h1, h2):
        if x == '?':
            continue
        if x == '\u2205':
            return False
        if x != y and y != '\u2205':
            return False
    return True


def fulfills(example, hypothesis):
    """True if `hypothesis` covers/matches `example`."""
    return all(h == '?' or h == e for h, e in zip(hypothesis, example))


def min_generalizations(h, example):
    """Return minimal generalizations of h that would make it cover `example`."""
    new_h = list(h)
    for i in range(len(h)):
        if h[i] == '\u2205':
            new_h[i] = example[i]
        elif h[i] != example[i]:
            new_h[i] = '?'
    return [tuple(new_h)]


def min_specializations(h, domains, example):
    """Return minimal specializations of h that would make it NOT cover `example`."""
    results = []
    for i in range(len(h)):
        if h[i] == '?':
            for val in domains[i]:
                if val != example[i]:
                    new_h = list(h)
                    new_h[i] = val
                    results.append(tuple(new_h))
        elif h[i] != '\u2205':
            new_h = list(h)
            new_h[i] = '\u2205'
            # (excluding the whole attribute this way isn't standard CE;
            #  we skip it -- specialization only replaces '?' slots)
    return results


def candidate_elimination(examples, labels):
    n_attrs = len(examples[0])
    domains = [sorted(set(row[i] for row in examples)) for i in range(n_attrs)]

    S = [tuple(['\u2205'] * n_attrs)]
    G = [tuple(['?'] * n_attrs)]

    print(f"Initial S: {S}")
    print(f"Initial G: {G}\n")

    for idx, (row, label) in enumerate(zip(examples, labels)):
        row = tuple(row)
        is_positive = str(label).strip().lower() in ("yes", "1", "true")
        print(f"--- Example {idx + 1}: {row}  [{label}] ---")

        if is_positive:
            # Remove G members that don't cover this positive example
            G = [g for g in G if fulfills(row, g)]

            new_S = []
            for s in S:
                if fulfills(row, s):
                    new_S.append(s)
                else:
                    for h in min_generalizations(s, row):
                        # Keep only if some g in G is more general than h
                        if any(more_general(g, h) for g in G):
                            new_S.append(h)
            # Remove hypotheses in S that are more general than another in S
            S = [s for s in new_S if not any(
                s != other and more_general(other, s) for other in new_S)]

        else:
            # Remove S members that incorrectly cover this negative example
            S = [s for s in S if not fulfills(row, s)]

            new_G = []
            for g in G:
                if not fulfills(row, g):
                    new_G.append(g)
                else:
                    for h in min_specializations(g, domains, row):
                        if any(more_general(h, s) for s in S):
                            new_G.append(h)
            # Remove hypotheses in G that are more specific than another in G
            G = [g for g in new_G if not any(
                g != other and more_general(other, g) for other in new_G)]

        print(f"  S = {S}")
        print(f"  G = {G}\n")

    return S, G


def demo_dataset():
    """Same EnjoySport dataset used for Find-S."""
    header = ["Sky", "AirTemp", "Humidity", "Wind", "Water", "Forecast"]
    examples = [
        ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same"],
        ["Sunny", "Warm", "High",   "Strong", "Warm", "Same"],
        ["Rainy", "Cold", "High",   "Strong", "Warm", "Change"],
        ["Sunny", "Warm", "High",   "Strong", "Cool", "Change"],
    ]
    labels = ["Yes", "Yes", "No", "Yes"]
    return header, examples, labels


if __name__ == "__main__":
    header, examples, labels = demo_dataset()

    print("Dataset (same as Find-S / Expt. 3):")
    print(f"{'  '.join(header):50s} Label")
    for row, label in zip(examples, labels):
        print(f"{'  '.join(row):50s} {label}")
    print()

    final_S, final_G = candidate_elimination(examples, labels)

    print("=" * 55)
    print("FINAL VERSION SPACE")
    print("=" * 55)
    print(f"Most Specific hypotheses (S): {final_S}")
    print(f"Most General hypotheses  (G): {final_G}")


Dataset (same as Find-S / Expt. 3):
Sky  AirTemp  Humidity  Wind  Water  Forecast      Label
Sunny  Warm  Normal  Strong  Warm  Same            Yes
Sunny  Warm  High  Strong  Warm  Same              Yes
Rainy  Cold  High  Strong  Warm  Change            No
Sunny  Warm  High  Strong  Cool  Change            Yes

Initial S: [('∅', '∅', '∅', '∅', '∅', '∅')]
Initial G: [('?', '?', '?', '?', '?', '?')]

--- Example 1: ('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')  [Yes] ---
  S = [('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')]
  G = [('?', '?', '?', '?', '?', '?')]

--- Example 2: ('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same')  [Yes] ---
  S = [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
  G = [('?', '?', '?', '?', '?', '?')]

--- Example 3: ('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change')  [No] ---
  S = [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
  G = [('Sunny', '?', '?', '?', '?', '?'), ('?', 'Warm', '?', '?', '?', '?'), ('?', '?', '?', '?', '?', '